<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b style="color: white;">IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p style="color: white;">Autumn Internship Program 2026</p>
<hr style="width:75%; background-color: white; border-color: white;">
<h3 style="color: white;">Project Title: Industrial Product Demand Forecasting</h3>
<h4 style="color: white;">Mentor Name: Dr. Chandan Biswas</h4>
</div>

### Question 1: Data Loading & Missing Value Justification (5 Marks)

Download the 'Historical Product Demand' dataset from Kaggle (`https://www.kaggle.com/datasets/felixzhao/productdemandforecasting`). Load the CSV file into a pandas DataFrame. Parse the 'Date' column as a datetime object. Identify missing values in the Date column and handle them. Print your written answer using Python `print()` statements to logically justify your chosen method for handling missing dates in a time-series context.

**Hint:** Use `pd.read_csv()` and `pd.to_datetime()`. Assess missing values. Use `print()` to state why dropping or imputing missing dates makes logical sense before forecasting.

**Expected Output:** A pandas DataFrame with cleaned data and a printed analytical justification.

In [1]:
import os
import pandas as pd
import kagglehub

# 1. Download the dataset from Kaggle and load the CSV
path = kagglehub.dataset_download("felixzhao/productdemandforecasting")
csv_path = os.path.join(path, "Historical Product Demand.csv")

df = pd.read_csv(csv_path)
print("Shape:", df.shape)

# 2. Parse the Date column (invalid/missing entries become NaT)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# 3. Assess missing values
total_rows = len(df)
missing_mask = df["Date"].isna()
missing_dates = missing_mask.sum()
print(f"\nTotal rows: {total_rows:,}")
print(f"Missing dates: {missing_dates:,} ({missing_dates / total_rows:.2%} of rows)")
print("\nMissing values per column:")
print(df.isna().sum())

print("\nWhere are the missing dates concentrated?")
print(df.loc[missing_mask, "Warehouse"].value_counts().to_string())
print(df.loc[missing_mask, "Product_Category"].value_counts().head(3).to_string())

# 4. Handle missing dates: drop, sort chronologically, reset index
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

print(f"\nRows after cleaning: {len(df):,}")
print(f"Remaining missing dates: {df['Date'].isna().sum()}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}\n")

# 5. Written justification
print("JUSTIFICATION FOR DROPPING MISSING DATES")
print("-" * 45)
print("1. The date is the index of a time series. A demand value with no date")
print("   cannot be placed in the sequence, so it cannot contribute to trend,")
print("   seasonality or autocorrelation.")
print("2. Imputing a date (forward fill, backward fill, interpolation or the")
print("   mean/median date) would invent information. Orders would be assigned")
print("   to days they may not have occurred on, creating fake demand spikes")
print("   and distorting the seasonal and lag patterns models rely on.")
print("3. The file is not ordered by time, so neighbouring rows say nothing")
print("   reliable about a missing date. Dates are not a smooth numeric")
print("   variable that can be estimated from adjacent values.")
print("4. The missing dates are about 1% of rows, so dropping them loses very")
print("   little data and is unlikely to bias the forecast.")
print("5. Dropping is the conservative choice: wrong timestamps silently")
print("   corrupt a model, whereas fewer observations only slightly reduce")
print("   the sample size.")
print("6. Caveat: all missing dates come from Warehouse A (mostly")
print("   Category_019), so Warehouse A demand is slightly under-represented.")
print("   This is worth noting, but the loss is small enough that dropping")
print("   remains the better option.")

print(df.head())

100%|██████████| 5.02M/5.02M [00:01<00:00, 3.73MB/s]

Extracting files...


Shape: (1048575, 5)

Total rows: 1,048,575
Missing dates: 11,239 (1.07% of rows)

Missing values per column:
Product_Code            0
Warehouse               0
Product_Category        0
Date                11239
Order_Demand            0
dtype: int64

Where are the missing dates concentrated?
Warehouse
Whse_A    11239
Product_Category
Category_019    10833
Category_028      290
Category_005       44

Rows after cleaning: 1,037,336
Remaining missing dates: 0
Date range: 2011-01-08 to 2017-01-09

JUSTIFICATION FOR DROPPING MISSING DATES
---------------------------------------------
1. The date is the index of a time series. A demand value with no date
   cannot be placed in the sequence, so it cannot contribute to trend,
   seasonality or autocorrelation.
2. Imputing a date (forward fill, backward fill, interpolation or the
   mean/median date) would invent information. Orders would be assigned
   to days they may not have occurred on, creating fake demand spikes
   and distorting the s

### Question 2: Target Variable Cleaning (5 Marks)

The target variable 'Order_Demand' contains non-numeric string characters (such as parentheses) in some rows. Remove all non-numeric characters from the 'Order_Demand' column and convert the column to an integer data type.

**Hint:** You can use the `.str.replace()` method with a regular expression (like `[^0-9]`) to strip out anything that is not a digit, followed by `.astype(int)`.

**Expected Output:** A pandas DataFrame where the 'Order_Demand' column is structured strictly as an integer data type.

In [2]:
import os
import pandas as pd
import kagglehub

# --- Question 1 steps (load, parse dates, drop missing dates) ---
path = kagglehub.dataset_download("felixzhao/productdemandforecasting")
df = pd.read_csv(os.path.join(path, "Historical Product Demand.csv"))
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# --- Question 2: clean the target variable ---
print("Before cleaning, dtype:", df["Order_Demand"].dtype)

# Rows that contain anything other than digits (e.g. "(100)" or trailing spaces)
has_non_digits = ~df["Order_Demand"].str.strip().str.fullmatch(r"\d+")
print(f"Rows with non-numeric characters: {has_non_digits.sum():,}")
print("Examples:", df.loc[has_non_digits, "Order_Demand"].str.strip().head(5).tolist())

# Remember which rows were written in brackets before the brackets are stripped
in_brackets = df["Order_Demand"].str.contains(r"\(", regex=True)

# Strip every non-digit character, then convert to integer
df["Order_Demand"] = (
    df["Order_Demand"]
    .str.replace(r"[^0-9]", "", regex=True)
    .astype(int)
)

# Verify the result
print("\nAfter cleaning, dtype:", df["Order_Demand"].dtype)
print("Missing values in Order_Demand:", df["Order_Demand"].isna().sum())
print("Min / Max:", df["Order_Demand"].min(), "/", df["Order_Demand"].max())

# Note on the brackets: in accounting notation "(100)" means minus 100 (a return or cancellation).
# The question asks for all non-numeric characters to be removed, so these values become positive.
total_unsigned = df["Order_Demand"].sum()
total_signed = df["Order_Demand"].where(~in_brackets, -df["Order_Demand"]).sum()
print(f"\nNOTE: {in_brackets.sum():,} values were written in brackets, e.g. (100), which normally means a negative quantity.")
print("Stripping the brackets, as the question asks, turns them into positive demand.")
print(f"This changes total demand by {(total_unsigned - total_signed) / total_unsigned:.2%} "
      f"({total_unsigned:,} instead of {total_signed:,}).")
print("To keep the sign instead: df['Order_Demand'].where(~in_brackets, -df['Order_Demand'])")
print()
print(df.dtypes)
print(df.head())

Using Colab cache for faster access to the 'productdemandforecasting' dataset.
Before cleaning, dtype: object
Rows with non-numeric characters: 5,899
Examples: ['(2)', '(2)', '(100)', '(400)', '(25)']

After cleaning, dtype: int64
Missing values in Order_Demand: 0
Min / Max: 0 / 4000000

NOTE: 5,899 values were written in brackets, e.g. (100), which normally means a negative quantity.
Stripping the brackets, as the question asks, turns them into positive demand.
This changes total demand by 0.59% (5,134,173,931 instead of 5,103,852,231).
To keep the sign instead: df['Order_Demand'].where(~in_brackets, -df['Order_Demand'])

Product_Code                object
Warehouse                   object
Product_Category            object
Date                datetime64[ns]
Order_Demand                 int64
dtype: object
   Product_Code Warehouse Product_Category       Date  Order_Demand
0  Product_0965    Whse_A     Category_006 2011-01-08             2
1  Product_1724    Whse_A     Category_003 2

### Question 3: Time-Series Aggregation (6 Marks)

Group the DataFrame by 'Date' and calculate the sum of 'Order_Demand' to simplify the multi-product dataset into a single temporal sequence. Store the result in a new DataFrame where the index represents the Date and a single column represents the aggregated daily demand.

**Hint:** Use the `.groupby()` method on the 'Date' column and apply `.sum()` to the 'Order_Demand' column. Ensure the result is a DataFrame rather than a pandas Series.

**Expected Output:** A time-series DataFrame indexed chronologically by date, containing a single numeric column for aggregate demand.

In [3]:
# Question 3: Aggregate Order_Demand into a single daily time series

# Double brackets keep the result as a DataFrame (a single brackets would give a Series)
daily_demand = df.groupby("Date")[["Order_Demand"]].sum()

print(type(daily_demand))
print("Shape:", daily_demand.shape)
print("Index sorted chronologically:", daily_demand.index.is_monotonic_increasing)
print(daily_demand.head())

<class 'pandas.core.frame.DataFrame'>
Shape: (1729, 1)
Index sorted chronologically: True
            Order_Demand
Date                    
2011-01-08             2
2011-05-31           108
2011-06-24         92000
2011-09-02          1250
2011-09-27            28


### Question 4: Advanced Time-Series Feature Engineering (8 Marks)

Generate chronological and statistical features from the aggregated daily demand DataFrame. Extract 'Year', 'Month', and 'DayOfWeek' from the datetime index. Additionally, create a 7-day lag feature ('Lag_7') and a 7-day rolling mean ('Rolling_Mean_7'). Drop any rows containing missing values resulting from these operations.

**Hint:** Use the `.index.year`, `.index.month`, and `.index.dayofweek` attributes. For statistical features, use `.shift(7)` and `.rolling(window=7).mean()`. Follow up with `.dropna()`.

**Expected Output:** A DataFrame expanded with multiple newly engineered chronological and statistical numerical features.

In [4]:
# Question 4: Chronological and statistical feature engineering

features = daily_demand.copy()

# Chronological features from the datetime index
features["Year"] = features.index.year
features["Month"] = features.index.month
features["DayOfWeek"] = features.index.dayofweek

# Statistical features (defined as the question specifies)
features["Lag_7"] = features["Order_Demand"].shift(7)
features["Rolling_Mean_7"] = features["Order_Demand"].rolling(window=7).mean()

# Drop the rows with missing values created by shift and rolling
features = features.dropna()

print("Shape:", features.shape)
print(features.dtypes)
print(features.head())

# --- Notes on the two statistical features ---
# Calendar days between each row and the row that Lag_7 takes its value from
gap_days = daily_demand.index.to_series().diff(7).dt.days.loc[features.index]
calendar_days = (daily_demand.index.max() - daily_demand.index.min()).days + 1

print("\nNOTES ON THE STATISTICAL FEATURES")
print(f"1. The series has {len(daily_demand):,} dates but spans {calendar_days:,} calendar days, so some days have no row.")
print("   shift(7) and rolling(7) therefore count rows (observed days), not calendar days. Only "
      f"{(gap_days == 7).mean():.0%} of rows take Lag_7")
print(f"   from exactly 7 days earlier (median gap {gap_days.median():.0f} days, largest gap {gap_days.max():.0f} days, in the sparse 2011 data).")
print("2. rolling(window=7).mean() includes the current day, so Rolling_Mean_7 contains one seventh of the value being")
print("   predicted (target leakage). A leak-free version uses only earlier days: shift(1).rolling(window=7).mean().")
print("   The definition requested in the question is kept here; Question 6 and Question 12 measure how much it matters.")

Shape: (1722, 6)
Order_Demand        int64
Year                int32
Month               int32
DayOfWeek           int32
Lag_7             float64
Rolling_Mean_7    float64
dtype: object
            Order_Demand  Year  Month  DayOfWeek    Lag_7  Rolling_Mean_7
Date                                                                     
2011-10-31             3  2011     10          0      2.0    14120.428571
2011-11-04           500  2011     11          4    108.0    14176.428571
2011-11-18          4162  2011     11          4  92000.0     1628.142857
2011-11-21           110  2011     11          0   1250.0     1465.285714
2011-11-22          5740  2011     11          1     28.0     2281.285714

NOTES ON THE STATISTICAL FEATURES
1. The series has 1,729 dates but spans 2,194 calendar days, so some days have no row.
   shift(7) and rolling(7) therefore count rows (observed days), not calendar days. Only 47% of rows take Lag_7
   from exactly 7 days earlier (median gap 8 days, largest ga

### Question 5: Outlier Detection and Analytical Reasoning (6 Marks)

Identify and remove outliers using the Z-score method. Calculate the Z-score for the daily demand feature. Filter the DataFrame to retain rows where the absolute Z-score is less than or equal to 3.0. Print your written answer using Python `print()` statements to explain the risk of retaining extreme outliers in a forecasting model.

**Hint:** Utilize `scipy.stats.zscore` or compute it manually. Apply boolean indexing to filter the rows. Output your reasoning via the print function.

**Expected Output:** A filtered DataFrame with anomaly days removed and a printed statement explaining the mathematical impact of outliers on machine learning algorithms.

In [5]:
# Question 5: Outlier detection using the Z-score method
from scipy import stats

# Z-score of the daily demand (distance from the mean in standard deviations)
z_scores = stats.zscore(features["Order_Demand"])

# Keep only rows where |Z| <= 3.0
mask = abs(z_scores) <= 3.0
features_clean = features[mask]

print(f"Rows before: {len(features):,}")
print(f"Rows after:  {len(features_clean):,}")
print(f"Outlier days removed: {(~mask).sum()}")
print(f"Std of demand before: {features['Order_Demand'].std():,.0f}")
print(f"Std of demand after:  {features_clean['Order_Demand'].std():,.0f}\n")

# Written reasoning
print("RISK OF RETAINING EXTREME OUTLIERS IN A FORECASTING MODEL")
print("-" * 58)
print("1. Outliers inflate the mean and the standard deviation, so the model")
print("   learns a distorted picture of what normal daily demand looks like.")
print("2. Most models minimise squared error (MSE). Squaring makes one extreme")
print("   day contribute far more to the loss than many normal days, so the")
print("   model bends its parameters to fit that single point.")
print("3. Linear and regression-based models are especially sensitive: one")
print("   extreme value can tilt the fitted line and shift all predictions.")
print("4. Rolling means and lag features carry the spike forward, so a single")
print("   anomaly contaminates the next several rows as well.")
print("5. Scaling methods (standardisation, min-max) are distorted by extreme")
print("   values, which squashes the normal range into a narrow band.")
print("6. The result is poor generalisation: the model may overpredict on")
print("   ordinary days and still fail to predict true spikes reliably.")
print("7. Caveat: if an extreme day is a genuine event (a promotion or")
print("   seasonal peak) removing it loses real signal, so outliers should be")
print("   checked against business context before they are discarded.")

features_clean.head()

Rows before: 1,722
Rows after:  1,718
Outlier days removed: 4
Std of demand before: 2,021,483
Std of demand after:  1,999,470

RISK OF RETAINING EXTREME OUTLIERS IN A FORECASTING MODEL
----------------------------------------------------------
1. Outliers inflate the mean and the standard deviation, so the model
   learns a distorted picture of what normal daily demand looks like.
2. Most models minimise squared error (MSE). Squaring makes one extreme
   day contribute far more to the loss than many normal days, so the
   model bends its parameters to fit that single point.
3. Linear and regression-based models are especially sensitive: one
   extreme value can tilt the fitted line and shift all predictions.
4. Rolling means and lag features carry the spike forward, so a single
   anomaly contaminates the next several rows as well.
5. Scaling methods (standardisation, min-max) are distorted by extreme
   values, which squashes the normal range into a narrow band.
6. The result is poor 

,Order_Demand,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,,
2011-10-31,3,2011,10,0,2.0,14120.428571
2011-11-04,500,2011,11,4,108.0,14176.428571
2011-11-18,4162,2011,11,4,92000.0,1628.142857
2011-11-21,110,2011,11,0,1250.0,1465.285714
2011-11-22,5740,2011,11,1,28.0,2281.285714


### Question 6: Exploratory Data Analysis and Correlation (7 Marks)

Compute the Pearson correlation matrix for all features and the target variable. Extract the correlation values specifically related to the demand target. Print your written answer using Python `print()` statements to identify the most correlated engineered feature and explain why this logical relationship exists.

**Hint:** Use the `.corr()` method on the DataFrame. Select the column corresponding to the target demand. Use `print()` to analyze the strongest temporal relationship.

**Expected Output:** A pandas Series showing correlation coefficients relative to the target and a printed analytical interpretation.

In [6]:
# Question 6: Pearson correlation with the target

corr_matrix = features_clean.corr(method="pearson")

# Correlation of every feature with the target, sorted from strongest to weakest
target_corr = corr_matrix["Order_Demand"].drop("Order_Demand")
target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

print("Correlation with Order_Demand:")
print(target_corr.round(3), "\n")

# Strongest overall, and strongest among the lag/rolling features
best_overall = target_corr.abs().idxmax()
best_stat = target_corr[["Lag_7", "Rolling_Mean_7"]].abs().idxmax()

# Leakage check: rebuild the rolling mean from the previous 7 observed days only (the current day is excluded)
leak_free_roll = daily_demand["Order_Demand"].shift(1).rolling(window=7).mean().loc[features_clean.index]
r_leak_free = features_clean["Order_Demand"].corr(leak_free_roll)
r_lag = target_corr["Lag_7"]
r_roll = target_corr["Rolling_Mean_7"]

# Share of total demand that falls on Saturday/Sunday
weekend_share = (features_clean.loc[features_clean["DayOfWeek"] >= 5, "Order_Demand"].sum()
                 / features_clean["Order_Demand"].sum())

print("INTERPRETATION")
print("-" * 40)
print(f"Strongest overall: {best_overall} (r = {target_corr[best_overall]:.3f})")
print(f"Strongest lag/rolling feature as constructed: {best_stat} (r = {target_corr[best_stat]:.3f})")
print(f"Same rolling mean without the current day: r = {r_leak_free:.3f}\n")

print(f"1. {best_stat} has the highest correlation of the statistical features, but much of that is mechanical.")
print("   The 7-row window includes the current day, so one seventh of the feature is the value being explained.")
print(f"   Once the current day is excluded the correlation falls from {r_roll:.3f} to {r_leak_free:.3f}, below Lag_7 ({r_lag:.3f}).")
print("   So the high value is mainly target leakage, not evidence that smoothing over seven days helps.")
print("2. Among the lag/rolling features, Lag_7 is the strongest relationship that is genuinely available at forecast time. Demand is autocorrelated and")
print("   follows a weekly cycle: the value from about a week earlier sits at a similar point of the weekly cycle, so it carries")
print("   the same weekday/weekend pattern. A 7-row average spans a whole weekly cycle and averages that pattern away; it mostly")
print("   measures the recent demand level.")
print(f"3. DayOfWeek has the largest correlation of all and it is negative (r = {target_corr['DayOfWeek']:.3f}). Weekdays are coded 0-4")
print(f"   and Saturday/Sunday 5-6, and weekend days carry only {weekend_share:.1%} of total demand, so a higher day number")
print("   means lower demand. This is weekly seasonality. (Pearson on a 0-6 code is a crude summary of a categorical effect.)")
print("4. Year and Month are weak. There is no strong linear long-term trend, and Month is cyclical, so a linear")
print("   correlation is a poor test for a seasonal pattern.")
print("5. Pearson correlation only captures linear relationships, so a weak value does not rule out a useful non-linear effect.")

target_corr

Correlation with Order_Demand:
DayOfWeek        -0.612
Rolling_Mean_7    0.416
Lag_7             0.355
Year              0.111
Month            -0.044
Name: Order_Demand, dtype: float64 

INTERPRETATION
----------------------------------------
Strongest overall: DayOfWeek (r = -0.612)
Strongest lag/rolling feature as constructed: Rolling_Mean_7 (r = 0.416)
Same rolling mean without the current day: r = 0.174

1. Rolling_Mean_7 has the highest correlation of the statistical features, but much of that is mechanical.
   The 7-row window includes the current day, so one seventh of the feature is the value being explained.
   Once the current day is excluded the correlation falls from 0.416 to 0.174, below Lag_7 (0.355).
   So the high value is mainly target leakage, not evidence that smoothing over seven days helps.
2. Among the lag/rolling features, Lag_7 is the strongest relationship that is genuinely available at forecast time. Demand is autocorrelated and
   follows a weekly cycle: the

,Order_Demand
DayOfWeek,-0.611881
Rolling_Mean_7,0.416074
Lag_7,0.355340
Year,0.110659
Month,-0.043799


### Question 7: Chronological Train-Test Split (5 Marks)

Split the dataset into training and testing sets while preserving the temporal order. Allocate the first 80% of the rows to the training set and the remaining 20% to the testing set. Separate the features into a matrix `X` and the target into a vector `y` for both subsets.

**Hint:** Avoid random splitting (do not use default `train_test_split` shuffling). Calculate the integer index corresponding to the 80% threshold and slice the DataFrames chronologically.

**Expected Output:** Four distinct variables containing the training features, testing features, training target, and testing target, strictly preserving temporal sequence.

In [7]:
# Question 7: Chronological 80/20 train-test split (no shuffling)

# Index that marks the 80% threshold
split_idx = int(len(features_clean) * 0.8)

# Slice by position so the temporal order is preserved
train = features_clean.iloc[:split_idx]
test = features_clean.iloc[split_idx:]

# Separate the features (X) from the target (y)
X_train = train.drop(columns="Order_Demand")
y_train = train["Order_Demand"]
X_test = test.drop(columns="Order_Demand")
y_test = test["Order_Demand"]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)
print("Train period:", X_train.index.min().date(), "to", X_train.index.max().date())
print("Test period: ", X_test.index.min().date(), "to", X_test.index.max().date())
print("Test starts after train ends:", X_train.index.max() < X_test.index.min())

X_train: (1374, 5) | y_train: (1374,)
X_test:  (344, 5) | y_test:  (344,)
Train period: 2011-10-31 to 2015-12-11
Test period:  2015-12-13 to 2017-01-09
Test starts after train ends: True


### Question 8: Feature Scaling and DataFrame Restoration (6 Marks)

Initialize a `StandardScaler`. Fit the scaler exclusively on the training features `X_train` and apply the transformation to both `X_train` and `X_test`. Explicitly convert the resulting scaled NumPy arrays back into pandas DataFrames, retaining the original column names and indices.

**Hint:** Import `StandardScaler`. Use `.fit_transform()` on training and `.transform()` on testing data. Reconstruct the DataFrame using `pd.DataFrame(data=..., index=..., columns=...)`.

**Expected Output:** Two pandas DataFrames containing standardized feature values, properly labeled with their original column headers.

In [8]:
# Question 8: Feature scaling and DataFrame restoration
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on the training features only, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert the NumPy arrays back to DataFrames with the original index and columns
X_train_scaled = pd.DataFrame(data=X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled = pd.DataFrame(data=X_test_scaled, index=X_test.index, columns=X_test.columns)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled: ", X_test_scaled.shape)
print("\nTrain feature means:\n", X_train_scaled.mean().round(3))
print("\nTrain feature std:\n", X_train_scaled.std(ddof=0).round(3))
print()
print(X_train_scaled.head())

X_train_scaled: (1374, 5)
X_test_scaled:  (344, 5)

Train feature means:
 Year             -0.0
Month             0.0
DayOfWeek         0.0
Lag_7             0.0
Rolling_Mean_7    0.0
dtype: float64

Train feature std:
 Year              1.0
Month             1.0
DayOfWeek         1.0
Lag_7             1.0
Rolling_Mean_7    1.0
dtype: float64

               Year    Month  DayOfWeek     Lag_7  Rolling_Mean_7
Date                                                             
2011-10-31 -2.08349  0.98172  -1.430360 -1.441343       -3.850052
2011-11-04 -2.08349  1.26750   0.584831 -1.441291       -3.849978
2011-11-18 -2.08349  1.26750   0.584831 -1.396485       -3.866381
2011-11-21 -2.08349  1.26750  -1.430360 -1.440734       -3.866594
2011-11-22 -2.08349  1.26750  -0.926562 -1.441330       -3.865527


### Question 9: Unsupervised Learning for Demand Regimes (7 Marks)

Perform unsupervised clustering to identify latent demand states. Initialize a K-Means algorithm with 3 clusters. Fit the algorithm on the scaled training features. Predict the cluster labels for both the training and testing sets, and append these labels as a new column named 'Demand_Regime' to the respective DataFrames.

**Hint:** Use `KMeans` from `sklearn.cluster`. After fitting, use the `.predict()` method. Assign the output arrays directly as a new column in your scaled training and testing DataFrames.

**Expected Output:** The training and testing DataFrames updated with a new categorical feature representing discrete cluster assignments.

In [9]:
# Question 9: K-Means clustering to identify demand regimes
from sklearn.cluster import KMeans

# Use only the original feature columns so the cell can be safely rerun
feature_cols = X_train.columns

# Fit K-Means with 3 clusters on the scaled training features
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_train_scaled[feature_cols])

# Predict the cluster labels and add them as a new column
X_train_scaled["Demand_Regime"] = kmeans.predict(X_train_scaled[feature_cols])
X_test_scaled["Demand_Regime"] = kmeans.predict(X_test_scaled[feature_cols])

print("Training regime counts:\n", X_train_scaled["Demand_Regime"].value_counts().sort_index())
print("\nTesting regime counts:\n", X_test_scaled["Demand_Regime"].value_counts().sort_index())
print()
print(X_train_scaled.head())

Training regime counts:
 Demand_Regime
0    301
1    538
2    535
Name: count, dtype: int64

Testing regime counts:
 Demand_Regime
0     38
1    306
Name: count, dtype: int64

               Year    Month  DayOfWeek     Lag_7  Rolling_Mean_7  \
Date                                                                
2011-10-31 -2.08349  0.98172  -1.430360 -1.441343       -3.850052   
2011-11-04 -2.08349  1.26750   0.584831 -1.441291       -3.849978   
2011-11-18 -2.08349  1.26750   0.584831 -1.396485       -3.866381   
2011-11-21 -2.08349  1.26750  -1.430360 -1.440734       -3.866594   
2011-11-22 -2.08349  1.26750  -0.926562 -1.441330       -3.865527   

            Demand_Regime  
Date                       
2011-10-31              2  
2011-11-04              2  
2011-11-18              2  
2011-11-21              2  
2011-11-22              2  


### Question 10: Baseline Supervised Model and Cross-Validation (8 Marks)

Instantiate a `RandomForestRegressor` with a fixed random state. Perform 5-fold Time Series cross-validation on the training set to compute the average Mean Absolute Error (MAE). Print the resulting average MAE.

**Hint:** Import `TimeSeriesSplit` and `cross_val_score` from `sklearn.model_selection`. Set the scoring parameter to `'neg_mean_absolute_error'`, convert the array to positive values, and calculate the mean.

**Expected Output:** A printed numerical value representing the average cross-validated Mean Absolute Error for the baseline model.

In [10]:
# Question 10: Baseline Random Forest with 5-fold time series cross-validation
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Baseline model with a fixed random state
rf_model = RandomForestRegressor(random_state=42)

# 5 chronological folds: each fold trains on the past and validates on the next block
tscv = TimeSeriesSplit(n_splits=5)

# Scikit-learn returns negative MAE, so flip the sign to get positive values
scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=tscv, scoring="neg_mean_absolute_error")
mae_scores = -scores

print("MAE for each fold:", mae_scores.round(0))
print(f"Average cross-validated MAE: {mae_scores.mean():,.2f}")

MAE for each fold: [859007. 808770. 755526. 739907. 936289.]
Average cross-validated MAE: 819,899.84


### Question 11: Hyperparameter Tuning (8 Marks)

Optimize the `RandomForestRegressor` using `GridSearchCV` combined with `TimeSeriesSplit`. Define a parameter grid searching across `n_estimators` (50, 100) and `max_depth` (None, 10). Fit the grid search on the training data and extract the optimal model configuration.

**Hint:** Define a dictionary mapping parameters to lists of values. Pass the model, grid, and cross-validator to `GridSearchCV`. Access the best hyperparameters using `.best_params_`.

**Expected Output:** An executed cell that tunes the model over the specified parameter space and prints the optimal hyperparameters.

In [11]:
# Question 11: Hyperparameter tuning with GridSearchCV + TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Parameter grid
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10],
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

# Fit on the training data
grid_search.fit(X_train_scaled, y_train)

# Optimal configuration
print("Best hyperparameters:", grid_search.best_params_)
print(f"Best cross-validated MAE: {-grid_search.best_score_:,.2f}")

best_model = grid_search.best_estimator_

Best hyperparameters: {'max_depth': 10, 'n_estimators': 50}
Best cross-validated MAE: 803,109.29


### Question 12: Tuned Model Evaluation and Analytical Reasoning (7 Marks)

Evaluate the optimized Random Forest model by generating predictions on the scaled testing features. Calculate the Mean Absolute Error (MAE) and the Root Mean Squared Error (RMSE). Print your written answer using Python `print()` statements to compare these test metrics against the baseline cross-validation score and determine if hyperparameter tuning yielded significant improvements.

**Hint:** Use `mean_absolute_error` and `mean_squared_error` (compute the square root). Structure a text response analyzing the magnitude of improvement and output it via `print()`.

**Expected Output:** Calculated error metrics and a printed written analysis objectively evaluating the impact of the tuning phase.

In [12]:
# Question 12: Evaluate the tuned model on the test set
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Tuned model predictions on the unseen test period
y_pred = best_model.predict(X_test_scaled)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Cross-validation scores for comparison
baseline_cv_mae = mae_scores.mean()           # from Question 10
tuned_cv_mae = -grid_search.best_score_       # from Question 11
cv_std = mae_scores.std(ddof=1)               # fold-to-fold spread of the baseline CV MAE

# Baseline model fitted on the full training set, for a like-for-like test comparison
baseline_model = RandomForestRegressor(random_state=42).fit(X_train_scaled, y_train)
baseline_pred = baseline_model.predict(X_test_scaled)
baseline_test_mae = mean_absolute_error(y_test, baseline_pred)

cv_gain = (baseline_cv_mae - tuned_cv_mae) / baseline_cv_mae * 100
test_gain = (baseline_test_mae - test_mae) / baseline_test_mae * 100
gap = (test_mae - baseline_cv_mae) / baseline_cv_mae * 100
gap_in_stds = (test_mae - baseline_cv_mae) / cv_std

# Simple reference forecasts
naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))     # always the training mean
weekday_mean = y_train.groupby(X_train["DayOfWeek"]).mean()
weekday_mae = mean_absolute_error(y_test, X_test["DayOfWeek"].map(weekday_mean))  # training average for that weekday

# Is the tuning gain statistically distinguishable? Paired test on the per-day absolute errors
p_tuning = stats.wilcoxon(np.abs(y_test.values - y_pred), np.abs(y_test.values - baseline_pred)).pvalue

# Does removing the period information (Year and the Year-driven Demand_Regime) lower the test error?
cols_no_period = [c for c in X_train_scaled.columns if c not in ("Year", "Demand_Regime")]
no_period_model = RandomForestRegressor(random_state=42, **grid_search.best_params_)
no_period_model.fit(X_train_scaled[cols_no_period], y_train)
no_period_mae = mean_absolute_error(y_test, no_period_model.predict(X_test_scaled[cols_no_period]))
no_period_change = (no_period_mae - test_mae) / test_mae * 100

# Leakage check: Rolling_Mean_7 rebuilt from the previous 7 observed days only (the current day is excluded)
leak_free = daily_demand["Order_Demand"].shift(1).rolling(window=7).mean()
rm_mean, rm_std = X_train["Rolling_Mean_7"].mean(), X_train["Rolling_Mean_7"].std(ddof=0)
Xtr_lf, Xte_lf = X_train_scaled.copy(), X_test_scaled.copy()
Xtr_lf["Rolling_Mean_7"] = (leak_free.loc[Xtr_lf.index] - rm_mean) / rm_std
Xte_lf["Rolling_Mean_7"] = (leak_free.loc[Xte_lf.index] - rm_mean) / rm_std
leak_free_model = RandomForestRegressor(random_state=42, **grid_search.best_params_).fit(Xtr_lf, y_train)
leak_free_mae = mean_absolute_error(y_test, leak_free_model.predict(Xte_lf))
leak_free_change = (leak_free_mae - test_mae) / test_mae * 100

# How concentrated are the errors?
sq_err = np.sort((y_test.values - y_pred) ** 2)[::-1]
top10_share = sq_err[:10].sum() / sq_err.sum() * 100

print(f"Tuned model test MAE:  {test_mae:,.0f}")
print(f"Tuned model test RMSE: {test_rmse:,.0f}\n")
print(f"Baseline CV MAE:       {baseline_cv_mae:,.0f}  (fold-to-fold std {cv_std:,.0f})")
print(f"Tuned CV MAE:          {tuned_cv_mae:,.0f}  ({cv_gain:.1f}% better than baseline)")
print(f"Baseline test MAE:     {baseline_test_mae:,.0f}")
print(f"Tuned test MAE:        {test_mae:,.0f}  ({test_gain:.1f}% better than baseline)\n")

print("Grid search, mean CV MAE for each setting:")
grid = pd.DataFrame(grid_search.cv_results_)
grid["mean_cv_mae"] = -grid["mean_test_score"]
print(grid[["param_n_estimators", "param_max_depth", "mean_cv_mae"]].round(0).to_string(index=False))

print("\nTest MAE of simple reference forecasts and ablations:")
print(f"  Always the training mean:                 {naive_mae:,.0f}")
print(f"  Training average for that weekday:        {weekday_mae:,.0f}")
print(f"  Tuned Random Forest:                      {test_mae:,.0f}")
print(f"  Tuned RF, leak-free Rolling_Mean_7:       {leak_free_mae:,.0f}  ({leak_free_change:+.1f}% vs tuned RF)")
print(f"  Tuned RF, without Year/Demand_Regime:     {no_period_mae:,.0f}  ({no_period_change:+.1f}% vs tuned RF)\n")

print("ANALYSIS OF THE TUNING PHASE")
print("-" * 40)
print(f"1. Tuning gave a small gain: {cv_gain:.1f}% lower MAE in cross-validation and {test_gain:.1f}% on the test set.")
print(f"   In cross-validation the gain ({baseline_cv_mae - tuned_cv_mae:,.0f}) is far smaller than the fold-to-fold spread ({cv_std:,.0f}),")
print(f"   so it is not significant there. On the test days a paired Wilcoxon test gives p = {p_tuning:.3f}, so the difference is")
print("   detectable, but it is small in practical terms. Conclusion: a modest improvement, not a major one.")
print("2. The grid shows max_depth=10 lowering the MAE by roughly 2% for both forest sizes, while 50 vs 100 trees changes it")
print("   by less than 0.3%. Limiting the depth probably reduces overfitting; the number of trees hardly matters.")
print("3. The grid was small (4 combinations), so only limited gains were possible. Larger gains would more likely come")
print("   from better features than from this grid.")
print(f"4. Test MAE is {gap:.1f}% above the baseline CV MAE, but that is only {gap_in_stds:.1f} fold standard deviations, and the last CV")
print(f"   fold alone scored {mae_scores[-1]:,.0f}. The gap is within normal fold-to-fold variation. Train and test targets are")
print(f"   similar (mean {y_train.mean():,.0f} vs {y_test.mean():,.0f}), and refitting without Year and Demand_Regime changes test MAE")
print(f"   by only {no_period_change:+.1f}%, so there is no evidence that unseen Year values or a period shift explain the gap.")
print(f"5. RMSE ({test_rmse:,.0f}) is always at least as large as MAE ({test_mae:,.0f}); the ratio of {test_rmse / test_mae:.2f} shows that")
print(f"   the errors vary in size. The 10 worst days out of {len(y_test)} account for {top10_share:.0f}% of the total squared error.")
print(f"6. Beating the training mean (MAE {naive_mae:,.0f}) is a weak test. Simply predicting the training average for each weekday")
print(f"   gives {weekday_mae:,.0f}, so the tuned model is only {(weekday_mae - test_mae) / weekday_mae * 100:.1f}% better than that.")
print("   Most of its skill is the weekly pattern.")
print("7. Target leakage: Rolling_Mean_7 includes the day being predicted. Replacing it with a version that uses only earlier")
print(f"   days raises the tuned model's test MAE to {leak_free_mae:,.0f} ({leak_free_change:+.1f}%), which is worse than the weekday average.")
print("   The accuracy reported above is therefore optimistic; in a real forecast the rolling mean must exclude the current day.")
print("8. Conclusion: tuning helped only slightly. The model is limited by its features, and part of its reported accuracy")
print("   comes from the leak rather than from genuine forecasting skill.")

Tuned model test MAE:  932,061
Tuned model test RMSE: 1,283,625

Baseline CV MAE:       819,900  (fold-to-fold std 80,222)
Tuned CV MAE:          803,109  (2.0% better than baseline)
Baseline test MAE:     965,777
Tuned test MAE:        932,061  (3.5% better than baseline)

Grid search, mean CV MAE for each setting:
 param_n_estimators param_max_depth  mean_cv_mae
                 50            None     822103.0
                100            None     819900.0
                 50              10     803109.0
                100              10     803261.0

Test MAE of simple reference forecasts and ablations:
  Always the training mean:                 1,538,011
  Training average for that weekday:        986,005
  Tuned Random Forest:                      932,061
  Tuned RF, leak-free Rolling_Mean_7:       1,063,554  (+14.1% vs tuned RF)
  Tuned RF, without Year/Demand_Regime:     926,178  (-0.6% vs tuned RF)

ANALYSIS OF THE TUNING PHASE
----------------------------------------
1. T

### Question 13: PyTorch Data Preparation and Skeleton Architecture (8 Marks)

Convert the scaled `X_train`, `X_test`, `y_train`, and `y_test` DataFrames/Series into PyTorch Tensors with a `torch.float32` data type. Reshape the target tensors into column vectors. Then, utilize the provided skeleton code to construct a Multi-Layer Perceptron (MLP) containing one hidden layer with 64 neurons, a ReLU activation function, and a final output layer. Fill in the missing architectural parameters.

**Hint:** Complete the `.tensor()` instantiations. Use `.unsqueeze(1)` or `.view(-1, 1)` for the targets. In the `MLP` class, fill in the `in_features` and `out_features` mapping to construct the network dimensions accurately.

**Expected Output:** Four correctly shaped PyTorch tensor objects and an instantiated object representing the completed MLP architecture.

In [13]:
import torch
import torch.nn as nn

# 1. Convert DataFrames/Series to PyTorch Tensors (Fill in the blanks)
# X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
# X_test_tensor = ...
# y_train_tensor = ... # Hint: reshape to a column vector
# y_test_tensor = ...
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)

# The target is in the millions (mean about 3 million), but the network starts with weights near 0 and Adam
# changes each weight by roughly the learning rate per step, so with raw targets it cannot get anywhere near
# that scale in 100 epochs and the loss stays flat. Standardise the target using the TRAINING mean and std only.
y_mean, y_std = y_train.mean(), y_train.std(ddof=0)
y_train_tensor = torch.tensor(((y_train - y_mean) / y_std).values, dtype=torch.float32).view(-1, 1)  # column vector
y_test_tensor = torch.tensor(((y_test - y_mean) / y_std).values, dtype=torch.float32).view(-1, 1)    # column vector

# 2. Complete the MLP Skeleton Architecture
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        # Define the hidden layer (64 neurons)
        self.hidden = nn.Linear(in_features=input_size, out_features=64)  # FILLED IN: 64
        # Define the output layer (yielding a single continuous value)
        self.output = nn.Linear(in_features=64, out_features=1)           # FILLED IN: 64 and 1
        # Define the activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass input through hidden layer
        x = self.hidden(x)
        # Apply activation function
        x = self.relu(x)
        # Pass through output layer
        x = self.output(x)
        return x

# 3. Instantiate the model
# input_size = X_train_tensor.shape[1]
# model = MLP(input_size=input_size)
input_size = X_train_tensor.shape[1]
model = MLP(input_size=input_size)

print("X_train_tensor:", X_train_tensor.shape, X_train_tensor.dtype)
print("X_test_tensor: ", X_test_tensor.shape, X_test_tensor.dtype)
print("y_train_tensor:", y_train_tensor.shape, y_train_tensor.dtype)
print("y_test_tensor: ", y_test_tensor.shape, y_test_tensor.dtype)
print(f"Target standardised with the training mean ({y_mean:,.0f}) and std ({y_std:,.0f})")
print()
print(model)

X_train_tensor: torch.Size([1374, 6]) torch.float32
X_test_tensor:  torch.Size([344, 6]) torch.float32
y_train_tensor: torch.Size([1374, 1]) torch.float32
y_test_tensor:  torch.Size([344, 1]) torch.float32
Target standardised with the training mean (2,952,444) and std (2,017,913)

MLP(
  (hidden): Linear(in_features=6, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
)


### Question 14: Multi-Layer Perceptron Training Loop (8 Marks)

Instantiate the MLP model. Define a Mean Squared Error (MSE) loss function and an Adam optimizer (learning rate = 0.001). Construct a training loop iterating over 100 epochs. Within each epoch, compute the forward pass, calculate the loss, execute a backward pass to compute gradients, and update the model weights. Print the loss every 20 epochs.

**Hint:** The standard order of operations inside the loop is: `optimizer.zero_grad()`, `model(inputs)`, `criterion(outputs, targets)`, `loss.backward()`, and `optimizer.step()`.

**Expected Output:** A printed log tracking the numerical reduction of the loss function over consecutive epochs.

In [14]:
# Question 14: Train the MLP

torch.manual_seed(42)  # reproducible weight initialisation

# Instantiate the model, loss function and optimizer
model = MLP(input_size=X_train_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# The target is standardised, so a model that always predicts the training mean has a loss of exactly 1.0
with torch.no_grad():
    initial_loss = criterion(model(X_train_tensor), y_train_tensor).item()
print(f"Loss before training: {initial_loss:.4f}  (always predicting the mean gives 1.0)")

# Training loop
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()                         # 1. clear old gradients
    outputs = model(X_train_tensor)               # 2. forward pass
    loss = criterion(outputs, y_train_tensor)     # 3. compute loss
    loss.backward()                               # 4. backward pass
    optimizer.step()                              # 5. update weights

    if epoch % 20 == 0:
        print(f"Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}")

Loss before training: 1.1796  (always predicting the mean gives 1.0)
Epoch [20/100], Loss: 0.7961
Epoch [40/100], Loss: 0.5762
Epoch [60/100], Loss: 0.4670
Epoch [80/100], Loss: 0.4258
Epoch [100/100], Loss: 0.4095


### Question 15: Business Impact Evaluation (6 Marks)

Evaluate the MLP predictions on the test set using a custom business penalty metric. In this supply chain scenario, under-predicting demand incurs a penalty equal to 2.0 times the absolute error, while over-predicting incurs a penalty of 1.0 times the absolute error. Calculate the total aggregated penalty. Print your written answer using Python `print()` statements to explain why under-prediction is typically penalized more severely in inventory management.

**Hint:** Calculate raw errors (`actual - predicted`). Apply conditional logic (e.g., boolean masks) to separate negative and positive errors, multiply by the specified weights, and compute the sum. Include the business justification via print.

**Expected Output:** A numerical calculation representing the simulated business cost of the forecasting errors and a printed logical justification for asymmetric penalties.

In [15]:
# Question 15: Business penalty metric (under-prediction costs 2x, over-prediction 1x)
import numpy as np

# MLP predictions on the test set. The network predicts standardised demand, so convert back to original units first.
model.eval()
with torch.no_grad():
    pred_scaled = model(X_test_tensor).numpy().flatten()

predictions = pred_scaled * y_std + y_mean
actual = y_test.values.astype(float)


def business_penalty(actual, predicted):
    """Under-prediction costs 2.0 x the absolute error, over-prediction costs 1.0 x."""
    errors = actual - predicted                  # positive = under-prediction, negative = over-prediction
    under_mask = errors > 0
    over_mask = errors < 0
    under_penalty = 2.0 * errors[under_mask].sum()
    over_penalty = 1.0 * np.abs(errors[over_mask]).sum()
    return under_penalty, over_penalty, under_mask.sum(), over_mask.sum()


under_penalty, over_penalty, n_under, n_over = business_penalty(actual, predictions)
total_penalty = under_penalty + over_penalty

print(f"Test predictions range from {predictions.min():,.0f} to {predictions.max():,.0f} (actual demand averages {actual.mean():,.0f})")
print(f"Under-predicted days: {n_under} | Penalty: {under_penalty:,.0f}")
print(f"Over-predicted days:  {n_over} | Penalty: {over_penalty:,.0f}")
print(f"Total business penalty: {total_penalty:,.0f}\n")

# Context: the same penalty for two reference forecasts on the same test days
rf_total = sum(business_penalty(actual, best_model.predict(X_test_scaled))[:2])
mean_total = sum(business_penalty(actual, np.full(len(actual), y_train.mean()))[:2])
print(f"For comparison: tuned Random Forest {rf_total:,.0f} | always the training mean {mean_total:,.0f}\n")

print("WHY UNDER-PREDICTION IS PENALISED MORE IN INVENTORY MANAGEMENT")
print("-" * 62)
print("1. Under-predicting demand causes stockouts. Customers who find the")
print("   product unavailable buy elsewhere, so the sale is lost permanently.")
print("2. Lost sales also damage customer loyalty and brand reputation, and a")
print("   customer lost today may not return, so the cost outlasts the order.")
print("3. Emergency fixes such as expedited shipping, overtime production or")
print("   spot-market purchases are far more expensive than normal replenishment.")
print("4. Over-predicting causes excess stock, which has a cost (storage,")
print("   capital tied up, possible markdowns) but the product can usually")
print("   still be sold later, so the loss is smaller and recoverable.")
print("5. In many supply chains the cost of a missed sale (lost margin plus")
print("   goodwill) is larger than the holding cost of surplus units, so the")
print("   asymmetric 2:1 weighting reflects the real economics.")
print("6. Optimising this metric pushes a model to forecast slightly high,")
print("   which is the safer direction when stockouts are the costlier risk.")
print("7. With a cost of 2 for a shortfall and 1 for a surplus, the cost-minimising forecast is the 2/3 quantile of demand,")
print("   not the mean (critical ratio 2 / (2 + 1)). Models trained with MSE aim at the mean, so training with an")
print("   asymmetric loss, or adding a safety-stock uplift to the forecast, would lower this penalty.")

Test predictions range from -137,830 to 5,912,141 (actual demand averages 3,022,122)
Under-predicted days: 128 | Penalty: 310,914,912
Over-predicted days:  216 | Penalty: 251,923,320
Total business penalty: 562,838,232

For comparison: tuned Random Forest 457,551,366 | always the training mean 805,598,046

WHY UNDER-PREDICTION IS PENALISED MORE IN INVENTORY MANAGEMENT
--------------------------------------------------------------
1. Under-predicting demand causes stockouts. Customers who find the
   product unavailable buy elsewhere, so the sale is lost permanently.
2. Lost sales also damage customer loyalty and brand reputation, and a
   customer lost today may not return, so the cost outlasts the order.
3. Emergency fixes such as expedited shipping, overtime production or
   spot-market purchases are far more expensive than normal replenishment.
4. Over-predicting causes excess stock, which has a cost (storage,
   capital tied up, possible markdowns) but the product can usually
   st